In [74]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import datetime as dt
import plotly.io as pio
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string

# import gensim
# from gensim.models import Word2Vec
# from gensim.models import KeyedVectors

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# pio.renderers.default = "browser"
pd.options.mode.chained_assignment = None
pd.set_option("display.max.rows", None)
pd.set_option("display.max.columns", None)

## Prediccion Box con Triage y Sintomas Parecidos


#### Preparar el set de datos


In [75]:
# Locally method
train_df = pd.read_csv("../dataset/raw/TRIAGE_2024.csv")
print(train_df.shape[0])
cols_rename = {"[ñ_MJ": "FECHA DE INGRESO", "A": "AISLADO"}
train_df.rename(columns=cols_rename, inplace=True)
train_df.loc[train_df["FECHA DE INGRESO"] == "01-01", "3+-99999|a"].iloc[
    0
] == train_df.loc[train_df["FECHA DE INGRESO"] == "01-02", "3+-99999|a"].iloc[0]
col_rename = {"3+-99999|a": "NUMERO DE TURNO"}
train_df.rename(columns=col_rename, inplace=True)
train_df = train_df[train_df["NUMERO DE TURNO"].str.contains("FECHA|N°") == False]
train_df = train_df[train_df["NOMBRE Y APELLIDO"].notnull()]
# train_df['TRIAGE'].dtype
train_df["TRIAGE"] = train_df["TRIAGE"].str.strip().str.upper()
vals_rename = {"I": 1, "II": 2, "III": 3, "IV": 4}
train_df["TRIAGE"] = train_df["TRIAGE"].replace(vals_rename)
train_df["TRIAGE"] = pd.to_numeric(
    train_df["TRIAGE"], downcast="signed", errors="coerce"
)
train_df["TRIAGE"].dtype
train_df["FECHA DE INGRESO"].dtype
train_df["FECHA DE INGRESO"] = train_df["FECHA DE INGRESO"] + "-2024"
train_df["FECHA DE INGRESO"] = pd.to_datetime(
    train_df["FECHA DE INGRESO"], format="%d-%m-%Y", errors="coerce"
)
train_df["FECHA DE INGRESO"].dtype
train_df["AISLADO"].unique()
data = {"AISLADO": [np.nan, "KPC", "NO", "SI", "No", "724", "Si", "A", "ECOLI METALO"]}
mapping = {
    np.nan: False,
    "NO": False,
    "No": False,
    "KPC": True,
    "ECOLI METALO": True,
    "SI": True,
    "Si": True,
    "A": True,
}
train_df["AISLADO"] = train_df["AISLADO"].map(mapping)
train_df = train_df[train_df["AISLADO"] != "724"]
train_df["AISLADO"] = train_df["AISLADO"].astype(bool)
train_df["DESTINO"].unique()
train_df = train_df[
    ~train_df["DESTINO"].isin(
        [
            "?",
            "INT",
            "HMD/ 315",
            "HMD",
            " ",
            "    ",
            "  ",
        ]
    )
]
train_df["ALTA"] = train_df["DESTINO"].str.contains(
    "alta|obito|traslado|derivacion|AL. VOL|DERIVAC", case=False, regex=True
)
# CUIDADO LA LINEA DEBAJO DE ESTE COMENTARIO BLOQUEA UN WARNING CON EL CAMBIO DE FILLNA
pd.set_option("future.no_silent_downcasting", True)
train_df = train_df.fillna({"ALTA": True}).infer_objects(copy=False)
cols_to_keep = [
    "NUMERO DE TURNO",
    "FECHA DE INGRESO",
    "NOMBRE Y APELLIDO",
    "MOTIVO DE CONSULTA",
    "BOX",
    "TRIAGE",
    "ENFERMERO",
    "MEDICO",
    "DESTINO",
    "ALTA",
    "AISLADO",
]
train_df = train_df[cols_to_keep]


def preprocess_text(text):
    # convert to lowercase
    text = str(text).lower()

    # remove punctuation
    text = "".join([char for char in text if char not in string.punctuation])

    # separate by syllables
    tokens = nltk.word_tokenize(text, "spanish")

    # eliminate common words without meaning
    stop_words = set(stopwords.words("spanish"))
    filtered_tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization (converting words to their base form)
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]
    return " ".join(lemmatized_tokens)


train_df["MOTIVO DE CONSULTA"] = train_df["MOTIVO DE CONSULTA"].apply(preprocess_text)

train_df.head()

2995


,NUMERO DE TURNO,FECHA DE INGRESO,NOMBRE Y APELLIDO,MOTIVO DE CONSULTA,BOX,TRIAGE,ENFERMERO,MEDICO,DESTINO,ALTA,AISLADO
1,1,2024-01-01,LUSI,fiebre tos,18,4.0,ERIKA,RODRIGO,NaN,True,False
2,2,2024-01-01,LO PINTO CARLOS,fiebre tos,17,4.0,SOLEDAD G,SOLEDAD,alta,True,False
3,3,2024-01-01,BANDERA,tos,12,4.0,ERIKA,RODRIGO,NaN,True,False
4,4,2024-01-01,TOMINO EDUARDO,hta,12,4.0,SOLEDAD G,SOLEDAD,ALTA,True,False
5,5,2024-01-01,D IORIO ROLANDO EMILIO,fiebre,5,4.0,ERIKA,SOLEDAD,NaN,True,False


### Preparar los datos especialmente para predecir el Box dependiendo el sintoma y el nivel de Triage


In [76]:
sintomas_rellenados_por_enfermeros = train_df["MOTIVO DE CONSULTA"].tolist()
resultados = []
sintomas_ideales = [
    "convulsiones",
    "trauma de cráneo",
    "dolor torácico",
    "dorsal",
    "dolor abdominal",
    "lumbar",
    "cefalea",
    "déficit motor",
    "disartria",
    "afasia",
    "pérdida aguda de visión",
    "disnea",
    "otro dolor en curso",
    "sobredosis de fármacos",
    "ingesta de tóxicos",
    "sangrado digestivo",
    "fiebre",
    "tos",
    "dt",
    "hta",
    "diarrea",
    "palpitaciones",
]
sintomas_sin_similitud = {}

# Create a DataFrame to store the results
df_relacion_pacientes_sintomas_random_forest = pd.DataFrame(
    columns=["Paciente", "Triage", "Sintoma real", "Box"] + sintomas_ideales
)

for index, row in train_df.iterrows():
    sintoma_paciente = [row["MOTIVO DE CONSULTA"]]
    textos = sintoma_paciente + sintomas_ideales
    vectorizer = TfidfVectorizer().fit_transform(textos)
    similitud_coseno = cosine_similarity(vectorizer)
    indices_ordenados = np.argsort(similitud_coseno[0][1:])[::-1]

    if similitud_coseno[0][indices_ordenados[0] + 1] == 0:
        if sintoma_paciente[0] in sintomas_sin_similitud:
            sintomas_sin_similitud[sintoma_paciente[0]] += 1
        else:
            sintomas_sin_similitud[sintoma_paciente[0]] = 1
    else:
        # Create a dictionary to store the patient's data
        paciente = {"Triage": train_df["TRIAGE"][index], "Box": train_df["BOX"][index]}
        # Add the similarity values for each ideal symptom
        for i in range(len(sintomas_ideales)):
            paciente[sintomas_ideales[i]] = similitud_coseno[0][i + 1]
        # Append the patient's data to the results list
        resultados.append(paciente)

df_relacion_pacientes_sintomas_random_forest = pd.DataFrame(resultados)

df_relacion_pacientes_sintomas_random_forest = (
    df_relacion_pacientes_sintomas_random_forest[
        df_relacion_pacientes_sintomas_random_forest["Triage"].notna()
    ]
)
df_relacion_pacientes_sintomas_random_forest["Box"] = pd.to_numeric(
    df_relacion_pacientes_sintomas_random_forest["Box"], errors="coerce"
).fillna(0)
df_relacion_pacientes_sintomas_random_forest = (
    df_relacion_pacientes_sintomas_random_forest.dropna()
)

display(df_relacion_pacientes_sintomas_random_forest.head())
if (df_relacion_pacientes_sintomas_random_forest == "C2").any().any():
    print("'c1' is in the DataFrame.")
else:
    print("'c1' is not in the DataFrame.")

sintomas_sin_similitud_ordenados = {
    k: v
    for k, v in sorted(
        sintomas_sin_similitud.items(), key=lambda item: item[1], reverse=True
    )
}
cantidad_de_sintmas = sum(sintomas_sin_similitud.values())
display(
    "Cantidad de sintomas sin alguna similitud exitosa: " + str(cantidad_de_sintmas)
)

,Triage,Box,convulsiones,trauma de cráneo,dolor torácico,dorsal,dolor abdominal,lumbar,cefalea,déficit motor,disartria,afasia,pérdida aguda de visión,disnea,otro dolor en curso,sobredosis de fármacos,ingesta de tóxicos,sangrado digestivo,fiebre,tos,dt,hta,diarrea,palpitaciones
0,4.0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,0.0
1,4.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,0.0
2,4.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0,0.0
3,4.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1.0,0.0,0.0
4,4.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,0.000000,0.0,0.0,0.0,0.0


'c1' is not in the DataFrame.


'Cantidad de sintomas sin alguna similitud exitosa: 701'

In [77]:
df_Random_Forest2 = df_relacion_pacientes_sintomas_random_forest.copy()
df_Random_Forest2 = pd.get_dummies(df_Random_Forest2, columns=["Box"])
df_Random_Forest2.head()

,Triage,convulsiones,trauma de cráneo,dolor torácico,dorsal,dolor abdominal,lumbar,cefalea,déficit motor,disartria,afasia,pérdida aguda de visión,disnea,otro dolor en curso,sobredosis de fármacos,ingesta de tóxicos,sangrado digestivo,fiebre,tos,dt,hta,diarrea,palpitaciones,Box_0.0,Box_1.0,Box_2.0,Box_3.0,Box_4.0,Box_5.0,Box_6.0,Box_7.0,Box_8.0,Box_9.0,Box_11.0,Box_12.0,Box_13.0,Box_14.0,Box_15.0,Box_16.0,Box_17.0,Box_18.0
0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,0.0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
1,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,0.0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0,0.0,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
3,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1.0,0.0,0.0,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
4,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,0.000000,0.0,0.0,0.0,0.0,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False


#### Chequeo que no haya ningun tipo sublistas en algun dato


In [78]:
for i in range(df_Random_Forest2.shape[1]):
    if isinstance(df_Random_Forest2.iloc[:, i].values[0], list):
        print(
            f"La característica {i} es una lista. Los valores son: {df_Random_Forest2.iloc[:, i].values[0]}"
        )
        # Encuentra los índices de las filas que contienen listas
        indices = [
            idx
            for idx, val in enumerate(df_Random_Forest2.iloc[:, i])
            if isinstance(val, list)
        ]
        # Elimina estas filas
        df_Random_Forest2 = df_Random_Forest2.drop(df_Random_Forest2.index[indices])

### Codigo de Entrenamiento del Modelo


In [113]:
print(df_Random_Forest2.shape[0])


box_columns = [col for col in df_Random_Forest2.columns if "Box" in col]

# Itera sobre cada columna 'Box'
for box_col in box_columns:
    # Crea un nuevo DataFrame para cada columna 'Box'
    df_temp = df_Random_Forest2.drop(box_columns, axis=1)
    df_temp["Target"] = df_Random_Forest2[box_col]

    # Separa las características (X) de la etiqueta objetivo (y)
    X_randomforest = df_temp.drop("Target", axis=1)
    y_randomforest = df_temp["Target"]

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_randomforest, y_randomforest, test_size=0.8, random_state=0
)

# Crear el modelo de árbol de decisión
clf = DecisionTreeClassifier()

# Entrenar el modelo
clf.fit(X_train, y_train)

# Hacer predicciones en el conjunto de prueba
y_pred = clf.predict(X_test)

# Calcular la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)

print(f"La precisión del modelo de árbol de decisión es: {accuracy}")

patient_data = X_randomforest.iloc[[1]]


# Asegúrate de que 'patient_data' tenga las mismas columnas que 'X_randomforest'
# assert set(patient_data.columns) == set(X_randomforest.columns)

# Usa el modelo para hacer una predicción
patient_box_prediction = clf.predict(patient_data)

print(f"El paciente iría al box: {patient_box_prediction}")

box_data = y_randomforest.iloc[1]
display(box_data)

648
La precisión del modelo de árbol de decisión es: 0.861271676300578
El paciente iría al box: [False]


False

In [80]:
# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_randomforest, y_randomforest, test_size=0.8, random_state=0
)

# Crear el modelo de árbol de decisión
clf = DecisionTreeClassifier()

# Entrenar el modelo
clf.fit(X_train, y_train)

# Hacer predicciones en el conjunto de prueba
y_pred = clf.predict(X_test)

# Calcular la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)

print(f"La precisión del modelo de árbol de decisión es: {accuracy}")

La precisión del modelo de árbol de decisión es: 0.861271676300578


### Codigo Funcion para predecir el Box


In [81]:
def PredecirBoxPaciente(patient_data):
    # Asegúrate de que patient_data tiene las mismas columnas que X_train
    assert set(patient_data.columns) == set(X_train.columns)
    return clf.predict(patient_data)

## Prediccion de Nivel de Tirage teniendo Sintomas similiares y Box


In [82]:
df = df_relacion_pacientes_sintomas_random_forest.copy()
display(df.head())
df = pd.get_dummies(df, columns=["Box"])
df = pd.get_dummies(df, columns=["Triage"])
df.head()

,Triage,Box,convulsiones,trauma de cráneo,dolor torácico,dorsal,dolor abdominal,lumbar,cefalea,déficit motor,disartria,afasia,pérdida aguda de visión,disnea,otro dolor en curso,sobredosis de fármacos,ingesta de tóxicos,sangrado digestivo,fiebre,tos,dt,hta,diarrea,palpitaciones
0,4.0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,0.0
1,4.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,0.0
2,4.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0,0.0
3,4.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1.0,0.0,0.0
4,4.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,0.000000,0.0,0.0,0.0,0.0


,convulsiones,trauma de cráneo,dolor torácico,dorsal,dolor abdominal,lumbar,cefalea,déficit motor,disartria,afasia,pérdida aguda de visión,disnea,otro dolor en curso,sobredosis de fármacos,ingesta de tóxicos,sangrado digestivo,fiebre,tos,dt,hta,diarrea,palpitaciones,Box_0.0,Box_1.0,Box_2.0,Box_3.0,Box_4.0,Box_5.0,Box_6.0,Box_7.0,Box_8.0,Box_9.0,Box_11.0,Box_12.0,Box_13.0,Box_14.0,Box_15.0,Box_16.0,Box_17.0,Box_18.0,Triage_0.0,Triage_1.0,Triage_2.0,Triage_3.0,Triage_4.0
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,0.0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.707107,0.707107,0.0,0.0,0.0,0.0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0,0.0,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1.0,0.0,0.0,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,0.000000,0.0,0.0,0.0,0.0,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True


In [83]:
# Obtiene todas las columnas que contienen 'Triage'
Triage_columns = [col for col in df.columns if "Triage" in col]


# Itera sobre cada columna 'Triage'
for triage_col in Triage_columns:
    # Crea un nuevo DataFrame para cada columna 'Triage'
    df_temp = df.drop(Triage_columns, axis=1)
    df_temp["Target"] = df[triage_col]

    # Separa las características (X) de la etiqueta objetivo (y)
    X = df_temp.drop("Target", axis=1)
    y = df_temp["Target"]
y.head()

0    True
1    True
2    True
3    True
4    True
Name: Target, dtype: bool

In [84]:
# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.8,
    random_state=0,
)

# Crear el modelo de árbol de decisión
clf = DecisionTreeClassifier()

# Entrenar el modelo
clf.fit(X_train, y_train)

# Hacer predicciones en el conjunto de prueba
y_pred = clf.predict(X_test)


accuracy = accuracy_score(y_test, y_pred)

print(f"La precisión del modelo de árbol de decisión es: {accuracy}")

La precisión del modelo de árbol de decisión es: 0.626204238921002


## Regresion Multivariable


In [85]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

box_columns = [col for col in df_Random_Forest2.columns if "Box" in col]

# Crea un nuevo DataFrame sin las columnas 'Box'
X_randomforest = df_Random_Forest2.drop(box_columns, axis=1)

# Usa las columnas 'Box' como tus etiquetas objetivo
y_randomforest = df_Random_Forest2[box_columns]

y_randomforest.head()

,Box_0.0,Box_1.0,Box_2.0,Box_3.0,Box_4.0,Box_5.0,Box_6.0,Box_7.0,Box_8.0,Box_9.0,Box_11.0,Box_12.0,Box_13.0,Box_14.0,Box_15.0,Box_16.0,Box_17.0,Box_18.0
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
4,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False


In [93]:
# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_randomforest, y_randomforest, test_size=0.8, random_state=0
)

# Crear el modelo de árbol de decisión
clf = DecisionTreeRegressor()

# Entrenar el modelo
clf.fit(X_train, y_train)

# Hacer predicciones en el conjunto de prueba
y_pred = clf.predict(X_test)

# Calcular el error cuadrático medio del modelo para cada variable objetivo
mse = [
    mean_squared_error(y_test.iloc[:, i], y_pred[:, i]) for i in range(y_test.shape[1])
]

print(
    f"El error cuadrático medio del modelo de árbol de decisión para cada variable objetivo es: {mse}"
)

El error cuadrático medio del modelo de árbol de decisión para cada variable objetivo es: [0.1607257546563905, 0.010276172125883108, 0.0038535645472061657, 0.004666425818882466, 0.023656604581460076, 0.13975540842541256, 0.06245664739884393, 0.05455884714193963, 0.038816634553628764, 0.06358381502890173, 0.011774780560907729, 0.054340077071290946, 0.05125061713831326, 0.06475794797687862, 0.0627910244059088, 0.14316687256148444, 0.1776312483888867, 0.1411202098051809]


In [109]:
# Asumamos que 'patient_data' es un DataFrame que contiene la información del paciente
patient_data = X_randomforest.iloc[[4]]


# Asegúrate de que 'patient_data' tenga las mismas columnas que 'X_randomforest'
# assert set(patient_data.columns) == set(X_randomforest.columns)

# Usa el modelo para hacer una predicción
patient_box_prediction = clf.predict(patient_data)

print(f"El paciente iría al box: {patient_box_prediction}")

box_data = y_randomforest.iloc[[4]]
display(box_data)

El paciente iría al box: [[0.   0.   0.   0.   0.   0.   0.5  0.   0.   0.   0.   0.   0.   0.
  0.   0.25 0.25 0.  ]]


,Box_0.0,Box_1.0,Box_2.0,Box_3.0,Box_4.0,Box_5.0,Box_6.0,Box_7.0,Box_8.0,Box_9.0,Box_11.0,Box_12.0,Box_13.0,Box_14.0,Box_15.0,Box_16.0,Box_17.0,Box_18.0
4,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False
